In [1]:
import pandas as pd
from pathlib import Path
raw_df = pd.read_csv(Path("../Data/AK_2000_overall_glacier_covered_area.csv"))
raw_df.head()

,RGIId,GLIMSId,BgnDate,EndDate,CenLon,CenLat,O1Region,O2Region,Area,Zmin,...,Lmax,Status,Connect,Form,TermType,Surging,Linkages,Name,GN_area,GN_comp_ye
0,RGI60-01.00002,G213332E63404N,20090703,-9999999,-146.668,63.404,1,2,0.558,1713,...,1197,0,0,0,0,9,9,NaN,0.5922,2000
1,RGI60-01.00003,G213920E63376N,20090703,-9999999,-146.080,63.376,1,2,1.685,1609,...,2106,0,0,0,0,9,9,NaN,1.8045,2000
2,RGI60-01.00004,G213880E63381N,20090703,-9999999,-146.120,63.381,1,2,3.681,1273,...,4175,0,0,0,0,9,9,NaN,3.6765,2000
3,RGI60-01.00005,G212943E63551N,20090703,-9999999,-147.057,63.551,1,2,2.573,1494,...,2981,0,0,0,0,9,9,NaN,2.6064,2000
4,RGI60-01.00006,G213756E63571N,20090703,-9999999,-146.244,63.571,1,2,10.470,1201,...,10518,0,0,0,0,9,9,NaN,7.7220,2000


In [2]:
#glaciers only
cleaning=raw_df[raw_df['Form'] == 0]
#clarifying column names and converting to Imperial 
cleaning['Id'] = cleaning['RGIId']
cleaning['Height (m)'] = cleaning['Zmax'] - cleaning['Zmin']
cleaning['Height (ft)'] = cleaning['Height (m)'] * 3.281
cleaning['Length (ft)'] = cleaning['Lmax'] * 3.281
cleaning['Area (mi^2)'] = cleaning['GN_area'] / 2.59
cleaning['Year'] = cleaning['GN_comp_ye']
#removing excess columns and sorting by Id
cleaning=cleaning.sort_values(by=["Id", "Year"]).reset_index()
cleaning=cleaning[['Id','Name','Year', 'CenLon', 'CenLat', 'Area (mi^2)', 'Height (ft)', 'Length (ft)']]
cleaning.head(25)

,Id,Name,Year,CenLon,CenLat,Area (mi^2),Height (ft),Length (ft)
0,RGI60-01.00002,NaN,2000,-146.668,63.404,0.228649,1414.111,3927.357
1,RGI60-01.00002,NaN,2002,-146.668,63.404,0.212317,1414.111,3927.357
2,RGI60-01.00002,NaN,2004,-146.668,63.404,0.213359,1414.111,3927.357
3,RGI60-01.00002,NaN,2006,-146.668,63.404,0.192162,1414.111,3927.357
4,RGI60-01.00002,NaN,2008,-146.668,63.404,0.236293,1414.111,3927.357
5,RGI60-01.00002,NaN,2010,-146.668,63.404,0.187645,1414.111,3927.357
6,RGI60-01.00002,NaN,2012,-146.668,63.404,0.194942,1414.111,3927.357
7,RGI60-01.00002,NaN,2014,-146.668,63.404,0.198417,1414.111,3927.357
8,RGI60-01.00002,NaN,2016,-146.668,63.404,0.216139,1414.111,3927.357
9,RGI60-01.00002,NaN,2018,-146.668,63.404,0.219614,1414.111,3927.357


In [3]:
cleaning.shape

(169443, 8)

In [4]:
namedonly=cleaning[cleaning['Name'].notna()]
namedonly.shape

(6243, 8)

In [5]:
#the loss of 10,000 entries is not worth having only named entries
cleaning.to_csv("../Data/AK_2000_overall_glacier_covered_area_cleaned.csv", index=False, header=True)

In [10]:
#a second csv with data over time for the ai to evaluate. thanks to chatgpt for some of the syntax.
def calculate_changes(df):
    changes = []
    for id_value, group in df.groupby('Id'):
        group_sorted = group.sort_values(by='Year')
        row = {'Id': id_value}
        for i in range(1, len(group_sorted)):
            prev_year = group_sorted.iloc[i-1]['Year']
            curr_year = group_sorted.iloc[i]['Year']
            
            area_change = group_sorted.iloc[i]['Area (mi^2)'] - group_sorted.iloc[i-1]['Area (mi^2)']
            long_change = group_sorted.iloc[i]['CenLon'] - group_sorted.iloc[i-1]['CenLon']
            lat_change = group_sorted.iloc[i]['CenLat'] - group_sorted.iloc[i-1]['CenLat']
            height_change = group_sorted.iloc[i]['Height (ft)'] - group_sorted.iloc[i-1]['Height (ft)']
            length_change = group_sorted.iloc[i]['Length (ft)'] - group_sorted.iloc[i-1]['Length (ft)']
            
            change_column_name = f"change between {prev_year} and {curr_year}"
            row[f"Longitude {change_column_name}"] = long_change
            row[f"Latitude {change_column_name}"] = lat_change
            row[f"Area (mi^2) {change_column_name}"] = area_change
            row[f"Height (ft) {change_column_name}"] = height_change
            row[f"Length (ft){change_column_name}"] = length_change
        changes.append(row)
    df_with_changes = pd.DataFrame(changes)
    return df_with_changes
over_time = calculate_changes(cleaning)
over_time.head(25)

,Id,Longitude change between 2000 and 2002,Latitude change between 2000 and 2002,Area (mi^2) change between 2000 and 2002,Height (ft) change between 2000 and 2002,Length (ft)change between 2000 and 2002,Longitude change between 2002 and 2004,Latitude change between 2002 and 2004,Area (mi^2) change between 2002 and 2004,Height (ft) change between 2002 and 2004,...,Longitude change between 2002 and 2014,Latitude change between 2002 and 2014,Area (mi^2) change between 2002 and 2014,Height (ft) change between 2002 and 2014,Length (ft)change between 2002 and 2014,Longitude change between 2002 and 2018,Latitude change between 2002 and 2018,Area (mi^2) change between 2002 and 2018,Height (ft) change between 2002 and 2018,Length (ft)change between 2002 and 2018
0,RGI60-01.00002,0.0,0.0,-0.016332,0.0,0.0,0.0,0.0,0.001042,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RGI60-01.00003,0.0,0.0,-0.034054,0.0,0.0,0.0,0.0,-0.053514,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RGI60-01.00004,0.0,0.0,-0.055946,0.0,0.0,0.0,0.0,-0.067413,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RGI60-01.00005,0.0,0.0,-0.004517,0.0,0.0,0.0,0.0,-0.033012,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RGI60-01.00006,0.0,0.0,-0.107027,0.0,0.0,0.0,0.0,0.545560,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RGI60-01.00007,0.0,0.0,-0.011467,0.0,0.0,0.0,0.0,0.012162,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RGI60-01.00008,0.0,0.0,-0.002432,0.0,0.0,0.0,0.0,-0.013900,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,RGI60-01.00009,0.0,0.0,-0.005560,0.0,0.0,0.0,0.0,-0.005560,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,RGI60-01.00010,0.0,0.0,-0.017722,0.0,0.0,0.0,0.0,-0.023977,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,RGI60-01.00011,0.0,0.0,-0.050039,0.0,0.0,0.0,0.0,-0.049691,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
over_time.shape

(16973, 276)

In [12]:
over_time.to_csv("../Data/AK_glacier_data_over_time.csv", index=False, header=True)